In [0]:
# AZURE DATABRICKS FINAL ASSESSMENT
## Healthcare Data Pipeline (PySpark + SQL + Delta + DLT + Unity Catalog)

In [0]:
from pyspark.sql.functions import *
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]
df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
df2 = df.withColumn("total_cost", col("consultation_fee") * col("tests_count"))
df2.show()

+--------+------------+---------+-----------+----------------+-----------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_cost|
+--------+------------+---------+-----------+----------------+-----------+----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      6000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     10000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      9000|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5000|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|      3000|
+--------+------------+---------+-----------+---------

In [0]:
df2.filter(col("total_cost") > 5000).show()

+--------+------------+---------+-----------+----------------+-----------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_cost|
+--------+------------+---------+-----------+----------------+-----------+----------+
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      6000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     10000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      9000|
+--------+------------+---------+-----------+----------------+-----------+----------+



In [0]:
dept_agg = df2.groupBy("department").agg(sum("total_cost").alias("total_revenue"),avg("total_cost").alias("avg_revenue"))
dept_agg.show()

+-----------+-------------+-----------------+
| department|total_revenue|      avg_revenue|
+-----------+-------------+-----------------+
| Cardiology|        20000|6666.666666666667|
|Orthopedics|        15000|           7500.0|
|Dermatology|         4500|           2250.0|
|  Neurology|         7000|           7000.0|
+-----------+-------------+-----------------+



In [0]:
dept_agg.orderBy(col("total_revenue").desc()).show()

+-----------+-------------+-----------------+
| department|total_revenue|      avg_revenue|
+-----------+-------------+-----------------+
| Cardiology|        20000|6666.666666666667|
|Orthopedics|        15000|           7500.0|
|  Neurology|         7000|           7000.0|
|Dermatology|         4500|           2250.0|
+-----------+-------------+-----------------+



In [0]:
df2.createOrReplaceTempView("patients")

In [0]:
%sql
SELECT * FROM patients WHERE department='Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total_cost
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5000
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000


In [0]:
%sql
SELECT city, SUM(total_cost) AS revenue FROM patients GROUP BY city;

city,revenue
Hyderabad,5000
Delhi,6000
Mumbai,1500
Bangalore,13000
Chennai,7000
Kolkata,9000
Ahmedabad,5000


In [0]:
%sql
SELECT patient_name, total_cost FROM patients ORDER BY total_cost DESC LIMIT 3;

patient_name,total_cost
Priya Nair,10000
Ananya Das,9000
Vikram Singh,7000


In [0]:
%sql
SELECT department, COUNT(*) AS count FROM patients GROUP BY department;

department,count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
df2.write.format("delta").mode("overwrite").saveAsTable("patients_delta")

In [0]:
%sql
SELECT * FROM patients_delta;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_cost
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,6000
103,Rahul Sharma,Mumbai,Dermatology,1500,1,1500
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
105,Vikram Singh,Chennai,Neurology,7000,1,7000
106,Ananya Das,Kolkata,Orthopedics,3000,3,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000
108,Meera Iyer,Bangalore,Dermatology,1500,2,3000


In [0]:
%sql
INSERT INTO patients_delta VALUES(109,"Rohit Jain","Pune","Cardiology",4000,2,8000);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE patients_delta SET consultation_fee = 6000, total_cost = 6000 * tests_count WHERE visit_id = 101;

num_affected_rows
1


In [0]:
spark.sql("""DELETE FROM patients_delta WHERE visit_id = 103""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
MERGE INTO patients_delta t
USING (
  SELECT 105 AS visit_id,
         'Vikram Singh' AS patient_name,
         'Chennai' AS city,
         'Neurology' AS department,
         8000 AS consultation_fee,
         2 AS tests_count,
         16000 AS total_cost
) s
ON t.visit_id = s.visit_id

WHEN MATCHED THEN UPDATE SET
  t.patient_name = s.patient_name,
  t.city = s.city,
  t.department = s.department,
  t.consultation_fee = s.consultation_fee,
  t.tests_count = s.tests_count,
  t.total_cost = s.total_cost

WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT * FROM patients_delta;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_cost
101,Arjun Reddy,Hyderabad,Cardiology,6000,1,6000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,6000
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
106,Ananya Das,Kolkata,Orthopedics,3000,3,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000
108,Meera Iyer,Bangalore,Dermatology,1500,2,3000
109,Rohit Jain,Pune,Cardiology,4000,2,8000
105,Vikram Singh,Chennai,Neurology,8000,2,16000


In [0]:
%sql
DESCRIBE HISTORY patients_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
11,2026-05-04T04:26:16.000Z,142604667681762,azuser6408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3992650955952955),6a40c497-ccb1-400c-b3b9-084051a9aa87,0504-040622-wbdrfqeb-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4608, p25FileSize -> 2556, numDeletionVectorsRemoved -> 1, minFileSize -> 2556, numAddedFiles -> 1, maxFileSize -> 2556, p75FileSize -> 2556, p50FileSize -> 2556, numAddedBytes -> 2556)",null,Databricks-Runtime/18.1.x-photon-scala2.13
10,2026-05-04T04:26:15.000Z,142604667681762,azuser6408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#13956L = cast(visit_id#13942 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3992650955952955),6a40c497-ccb1-400c-b3b9-084051a9aa87,0504-040622-wbdrfqeb-v2n,9,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2052, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3495, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1232, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2231)",null,Databricks-Runtime/18.1.x-photon-scala2.13
9,2026-05-04T04:24:22.000Z,142604667681762,azuser6408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3992650955952955),65f335c9-6b77-4ea4-bbc9-cfb74aa4da5c,0504-040622-wbdrfqeb-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4608, p25FileSize -> 2556, numDeletionVectorsRemoved -> 1, minFileSize -> 2556, numAddedFiles -> 1, maxFileSize -> 2556, p75FileSize -> 2556, p50FileSize -> 2556, numAddedBytes -> 2556)",null,Databricks-Runtime/18.1.x-photon-scala2.13
8,2026-05-04T04:24:21.000Z,142604667681762,azuser6408_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#13399L = cast(visit_id#13385 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3992650955952955),65f335c9-6b77-4ea4-bbc9-cfb74aa4da5c,0504-040622-wbdrfqeb-v2n,7,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2052, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3350, materializeSourceTimeMs -> 2, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1220, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2084)",null,Databricks-Runtime/18.1.x-photon-scala2.13
7,2026-05-04T04:23:57.000Z,142604667681762,azuser6408_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3992650955952955),7edca6f7-4e98-49a3-9

In [0]:
%sql
SELECT * FROM patients_delta VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_cost
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,6000
103,Rahul Sharma,Mumbai,Dermatology,1500,1,1500
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
105,Vikram Singh,Chennai,Neurology,7000,1,7000
106,Ananya Das,Kolkata,Orthopedics,3000,3,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000
108,Meera Iyer,Bangalore,Dermatology,1500,2,3000


In [0]:
%sql
VACUUM patients_delta RETAIN 168 HOURS DRY RUN;

path


In [0]:
df.write.mode("overwrite").saveAsTable("patients_parquet")

In [0]:
%sql
CONVERT TO DELTA patients_parquet;

In [0]:
%sql
SELECT * FROM patients_parquet;

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,1
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
103,Rahul Sharma,Mumbai,Dermatology,1500,1
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2


In [0]:
df2.write.format("delta").mode("overwrite").saveAsTable("incremental_patients")

In [0]:
updates = [
(105,"Vikram Singh","Chennai","Neurology",9000,2,18000),  # existing → update
(110,"New Patient","Delhi","Cardiology",4000,1,4000)      # new → insert
]

updates_df = spark.createDataFrame(updates, df2.columns)
updates_df.createOrReplaceTempView("updates")

In [0]:
%sql
MERGE INTO incremental_patients t
USING updates s
ON t.visit_id = s.visit_id

WHEN MATCHED THEN UPDATE SET
  t.patient_name = s.patient_name,
  t.city = s.city,
  t.department = s.department,
  t.consultation_fee = s.consultation_fee,
  t.tests_count = s.tests_count,
  t.total_cost = s.total_cost

WHEN NOT MATCHED THEN INSERT (
  visit_id, patient_name, city, department,
  consultation_fee, tests_count, total_cost
)
VALUES (
  s.visit_id, s.patient_name, s.city, s.department,
  s.consultation_fee, s.tests_count, s.total_cost
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM incremental_patients;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_cost
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,6000
103,Rahul Sharma,Mumbai,Dermatology,1500,1,1500
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
106,Ananya Das,Kolkata,Orthopedics,3000,3,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000
108,Meera Iyer,Bangalore,Dermatology,1500,2,3000
105,Vikram Singh,Chennai,Neurology,9000,2,18000
110,New Patient,Delhi,Cardiology,4000,1,4000


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS healthcare_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_catalog.hospital_schema;

In [0]:
%sql
CREATE TABLE healthcare_catalog.hospital_schema.patients_uc USING DELTA AS SELECT * FROM patients_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW TABLES IN healthcare_catalog.hospital_schema;

database,tableName,isTemporary
hospital_schema,patients_uc,false
,patients,true
,updates,true


In [0]:
%sql
CREATE TABLE healthcare_catalog.hospital_schema.revenue_summary AS
SELECT department, SUM(total_cost) AS total_revenue
FROM healthcare_catalog.hospital_schema.patients_uc
GROUP BY department;

num_affected_rows,num_inserted_rows


In [0]:
%sql
GRANT SELECT ON TABLE healthcare_catalog.hospital_schema.patients_uc 
TO `account users`;

Lineage: Available in Databricks UI → Data → Lineage tab

Audit Logs:
- Available via system tables or admin console
- Used to track access and operations

Final Capstone — Healthcare Data Pipeline A complete end-to-end healthcare data pipeline was built using PySpark, SQL, Delta Lake, Delta Live Tables (DLT), and Unity Catalog. The pipeline follows a structured architecture from raw data ingestion to analytics and governance: Raw Data → Bronze → Silver → Gold → Delta Tables → Unity Catalog. Initially, data was created and transformed using PySpark, including the addition of a derived column *total_cost* (consultation_fee × tests_count). SQL was used for querying, aggregation, and generating insights. Delta Lake was implemented to ensure reliable storage with ACID properties, supporting operations such as INSERT, UPDATE, DELETE, and MERGE for incremental data processing. Incremental loading was handled using MERGE logic to update existing records and insert new ones efficiently. Advanced Delta features like Time Travel, DESCRIBE HISTORY, and VACUUM (dry run) were used for versioning and storage management. A DLT pipeline was created with Bronze (raw data), Silver (cleaned and transformed data), and Gold (aggregated insights) layers to automate data flow. Unity Catalog was used for governance by creating catalogs and schemas, managing tables, applying access control, and observing data lineage. Business insights revealed that the Cardiology department generates the highest revenue, cities like Bangalore and Chennai contribute significantly, and higher test counts lead to increased revenue. Overall, the pipeline demonstrates a scalable, reliable, and governed data engineering solution.

DLT pipeline was implemented using Bronze, Silver, and Gold layers.
The pipeline was executed using Delta Live Tables (DLT) within Databricks.